In [4]:
from __future__ import annotations

"""
Banking Analytics Prompt Library - Gradio App

Excel-integrated version.

Expected Excel columns:
- Prompt ID
- Prompt Name
- Category
- Tags/Labels
- Prompt Objective
- Prompt
- Required Inputs
- Sample Output
"""

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
import statistics
import textwrap
from pathlib import Path

import pandas as pd
import gradio as gr
from gradio.themes.utils import colors, sizes


# ============================================================
# Data Models
# ============================================================

@dataclass
class PromptInput:
    name: str
    description: str


@dataclass
class PromptRecord:
    id: int
    name: str
    category: str
    tags: List[str]
    prompt_objective: str
    prompt_template: str
    required_inputs: List[PromptInput]
    optional_inputs: List[PromptInput]
    expected_result: str
    ratings: List[int] = field(default_factory=list)
    comments: List[str] = field(default_factory=list)


# ============================================================
# Excel Loader
# ============================================================

EXCEL_FILE_PATH = "prompt_guide.xlsx"


class PromptExcelLoader:
    """
    Loads prompts from Excel and maps them into PromptRecord objects.
    """

    REQUIRED_COLUMNS = [
        "Prompt ID",
        "Prompt Name",
        "Category",
        "Tags/Labels",
        "Prompt Objective",
        "Prompt",
        "Required Inputs",
        "Sample Output",
    ]

    @classmethod
    def load_from_excel(cls, file_path: str) -> List[PromptRecord]:
        path = Path(file_path)
        if not path.exists():
            return []

        df = pd.read_excel(path)
        df.columns = [str(col).strip() for col in df.columns]

        missing = [col for col in cls.REQUIRED_COLUMNS if col not in df.columns]
        if missing:
            raise ValueError(f"Missing required Excel columns: {', '.join(missing)}")

        records: List[PromptRecord] = []
        for _, row in df.iterrows():
            record = cls._row_to_prompt_record(row)
            if record is not None:
                records.append(record)

        return records

    @classmethod
    def _row_to_prompt_record(cls, row: pd.Series) -> Optional[PromptRecord]:
        prompt_id = cls._safe_int(row.get("Prompt ID"))
        prompt_name = cls._safe_text(row.get("Prompt Name"))

        if prompt_id is None or not prompt_name:
            return None

        category = cls._safe_text(row.get("Category")) or "Uncategorized"
        tags = cls._parse_tags(row.get("Tags/Labels"))
        prompt_objective = cls._safe_text(row.get("Prompt Objective"))
        prompt_template = cls._safe_text(row.get("Prompt"))
        sample_output = cls._safe_text(row.get("Sample Output"))

        required_inputs, optional_inputs = cls._parse_inputs(
            cls._safe_text(row.get("Required Inputs"))
        )

        return PromptRecord(
            id=prompt_id,
            name=prompt_name,
            category=category,
            tags=tags,
            prompt_objective=prompt_objective,
            prompt_template=prompt_template,
            required_inputs=required_inputs,
            optional_inputs=optional_inputs,
            expected_result=sample_output,
        )

    @staticmethod
    def _safe_text(value: Any) -> str:
        if pd.isna(value):
            return ""
        return str(value).strip()

    @staticmethod
    def _safe_int(value: Any) -> Optional[int]:
        if pd.isna(value):
            return None
        try:
            return int(value)
        except (TypeError, ValueError):
            return None

    @staticmethod
    def _parse_tags(value: Any) -> List[str]:
        text = PromptExcelLoader._safe_text(value)
        if not text:
            return []

        normalized = text
        for sep in [",", ";", "|", "/"]:
            normalized = normalized.replace(sep, ",")

        return [item.strip() for item in normalized.split(",") if item.strip()]

    @staticmethod
    def _parse_inputs(value: str) -> Tuple[List[PromptInput], List[PromptInput]]:
        """
        Supports simple parsing from one Excel cell.

        Accepted forms:
        - one item per line
        - comma-separated values
        - lines beginning with Required: or Optional:
        """
        if not value:
            return [], []

        required_inputs: List[PromptInput] = []
        optional_inputs: List[PromptInput] = []

        lines = [line.strip() for line in value.splitlines() if line.strip()]
        if not lines:
            lines = [item.strip() for item in value.split(",") if item.strip()]

        default_bucket = required_inputs

        for raw_line in lines:
            line = raw_line.lstrip("-*• ").strip()
            lower_line = line.lower()

            if lower_line.startswith("required:"):
                items = [item.strip() for item in line.split(":", 1)[1].split(",") if item.strip()]
                required_inputs.extend(
                    [PromptInput(name=item, description="Provided by user") for item in items]
                )
                default_bucket = required_inputs
                continue

            if lower_line.startswith("optional:"):
                items = [item.strip() for item in line.split(":", 1)[1].split(",") if item.strip()]
                optional_inputs.extend(
                    [PromptInput(name=item, description="Optional user input") for item in items]
                )
                default_bucket = optional_inputs
                continue

            default_bucket.append(PromptInput(name=line, description="Provided by user"))

        return required_inputs, optional_inputs


# ============================================================
# Repository Layer
# ============================================================

class PromptRepository:
    def __init__(self, seed_data: Optional[List[PromptRecord]] = None):
        self._prompts: Dict[int, PromptRecord] = {}
        if seed_data:
            for prompt in seed_data:
                self._prompts[prompt.id] = prompt

    def list_all(self) -> List[PromptRecord]:
        return list(self._prompts.values())

    def get_by_id(self, prompt_id: int) -> Optional[PromptRecord]:
        return self._prompts.get(prompt_id)

    def search(self, search_text: str = "", category: str = "All") -> List[PromptRecord]:
        search_text = (search_text or "").strip().lower()

        results = []
        for prompt in self._prompts.values():
            if category != "All" and prompt.category != category:
                continue

            haystack = " ".join([
                prompt.name,
                prompt.category,
                " ".join(prompt.tags),
                prompt.prompt_objective,
                prompt.expected_result,
                prompt.prompt_template,
            ]).lower()

            if not search_text or search_text in haystack:
                results.append(prompt)

        return sorted(results, key=lambda p: p.name.lower())

    def add_rating(self, prompt_id: int, rating: int) -> bool:
        prompt = self.get_by_id(prompt_id)
        if not prompt:
            return False
        prompt.ratings.append(rating)
        return True

    def add_comment(self, prompt_id: int, comment: str) -> bool:
        prompt = self.get_by_id(prompt_id)
        if not prompt:
            return False
        prompt.comments.append(comment)
        return True

    def categories(self) -> List[str]:
        categories = sorted({prompt.category for prompt in self._prompts.values()})
        return ["All"] + categories


# ============================================================
# Service Layer
# ============================================================

class PromptService:
    def __init__(self, repository: PromptRepository):
        self.repository = repository

    def get_prompt_choices(self, search_text: str, category: str) -> List[Tuple[str, int]]:
        prompts = self.repository.search(search_text, category)
        return [(f"{p.name} [{p.category}]", p.id) for p in prompts]

    def get_prompt_detail(self, prompt_id: int) -> Dict[str, str]:
        prompt = self.repository.get_by_id(prompt_id)
        if not prompt:
            return {
                "name": "",
                "category": "",
                "tags": "",
                "objective": "",
                "required_inputs": "",
                "optional_inputs": "",
                "expected_result": "",
                "rating_summary": "No ratings yet",
                "comments": "No comments yet",
                "visualizer": "",
                "copy_payload": "",
            }

        avg_rating = round(statistics.mean(prompt.ratings), 2) if prompt.ratings else None

        required_inputs = "\n".join(
            [f"• {inp.name}: {inp.description}" for inp in prompt.required_inputs]
        ) or "None"

        optional_inputs = "\n".join(
            [f"• {inp.name}: {inp.description}" for inp in prompt.optional_inputs]
        ) or "None"

        comments = "\n".join([f"• {c}" for c in prompt.comments]) or "No comments yet"

        rating_summary = (
            f"Average Rating: {avg_rating}/5 from {len(prompt.ratings)} review(s)"
            if avg_rating is not None
            else "No ratings yet"
        )

        visualizer = self.build_prompt_visualizer(prompt)
        copy_payload = self.build_copy_payload(prompt)

        return {
            "name": prompt.name,
            "category": prompt.category,
            "tags": ", ".join(prompt.tags),
            "objective": prompt.prompt_objective,
            "required_inputs": required_inputs,
            "optional_inputs": optional_inputs,
            "expected_result": prompt.expected_result,
            "rating_summary": rating_summary,
            "comments": comments,
            "visualizer": visualizer,
            "copy_payload": copy_payload,
        }

    def build_prompt_visualizer(self, prompt: PromptRecord) -> str:
        required = ", ".join([inp.name for inp in prompt.required_inputs]) or "None"
        optional = ", ".join([inp.name for inp in prompt.optional_inputs]) or "None"

        return textwrap.dedent(f"""
        {prompt.name}

        Category: {prompt.category}
        Tags: {', '.join(prompt.tags)}
        Objective: {prompt.prompt_objective or 'Not provided'}

        Inputs
        - Required: {required}
        - Optional: {optional}

        Prompt
        {prompt.prompt_template}

        Sample Output
        {prompt.expected_result}
        """).strip()

    def build_copy_payload(self, prompt: PromptRecord) -> str:
        required_help = "\n".join(
            [f"• {inp.name}: {inp.description}" for inp in prompt.required_inputs]
        ) or "• None"

        optional_help = "\n".join(
            [f"• {inp.name}: {inp.description}" for inp in prompt.optional_inputs]
        ) or "• None"

        return textwrap.dedent(f"""
        Prompt ID: {prompt.id}
        Prompt Name: {prompt.name}
        Category: {prompt.category}
        Tags: {', '.join(prompt.tags)}
        Prompt Objective: {prompt.prompt_objective}

        Required Inputs:
        {required_help}

        Optional Inputs:
        {optional_help}

        Prompt Template:
        {prompt.prompt_template}

        Sample Output:
        {prompt.expected_result}
        """).strip()

    def submit_rating(self, prompt_id: int, rating: int) -> str:
        if not prompt_id:
            return "Please select a prompt before rating."
        if rating < 1 or rating > 5:
            return "Rating must be between 1 and 5."

        success = self.repository.add_rating(prompt_id, rating)
        return "Rating submitted successfully." if success else "Unable to submit rating."

    def submit_comment(self, prompt_id: int, comment: str) -> str:
        if not prompt_id:
            return "Please select a prompt before commenting."
        if not comment or not comment.strip():
            return "Comment cannot be empty."

        success = self.repository.add_comment(prompt_id, comment.strip())
        return "Comment added successfully." if success else "Unable to add comment."


# ============================================================
# Fallback Seed Data
# ============================================================

def get_seed_prompts() -> List[PromptRecord]:
    return [
        PromptRecord(
            id=1,
            name="Credit Risk Summary Generator",
            category="Risk Analytics",
            tags=["credit-risk", "portfolio", "summary", "banking"],
            prompt_objective="Summarize credit portfolio performance and surface management actions.",
            prompt_template=(
                "You are a banking analytics expert. Analyze the following credit portfolio data: "
                "{portfolio_data}. Focus on delinquency trends, segment concentration, early warning signals, "
                "and recommended risk actions for management."
            ),
            required_inputs=[
                PromptInput("portfolio_data", "Portfolio snapshot, performance metrics, and delinquency measures."),
            ],
            optional_inputs=[
                PromptInput("time_period", "Reporting month, quarter, or year for comparison."),
                PromptInput("region", "Geography or branch segmentation if relevant."),
            ],
            expected_result="A concise risk summary with trends, red flags, and management recommendations.",
            ratings=[5, 4],
            comments=["Very useful for portfolio review meetings."],
        )
    ]


# ============================================================
# UI Helpers
# ============================================================

def build_prompt_dropdown_choices(service: PromptService, search_text: str, category: str):
    return service.get_prompt_choices(search_text, category)


# ============================================================
# Controller Functions
# ============================================================

def refresh_prompt_list(search_text: str, category: str):
    choices = build_prompt_dropdown_choices(app_service, search_text, category)
    selected_value = choices[0][1] if choices else None

    return (
        gr.update(choices=choices, value=selected_value),
        *render_prompt_details(selected_value),
    )


def render_prompt_details(prompt_id: Optional[int]):
    detail = app_service.get_prompt_detail(prompt_id) if prompt_id else app_service.get_prompt_detail(-1)

    return (
        detail["name"],
        detail["category"],
        detail["tags"],
        detail["objective"],
        detail["required_inputs"],
        detail["optional_inputs"],
        detail["expected_result"],
        detail["rating_summary"],
        detail["comments"],
        detail["visualizer"],
        detail["copy_payload"],
    )


def handle_rating(prompt_id: Optional[int], rating: int):
    message = app_service.submit_rating(prompt_id, int(rating))
    detail = app_service.get_prompt_detail(prompt_id) if prompt_id else app_service.get_prompt_detail(-1)
    return message, detail["rating_summary"]


def handle_comment(prompt_id: Optional[int], comment: str):
    message = app_service.submit_comment(prompt_id, comment)
    detail = app_service.get_prompt_detail(prompt_id) if prompt_id else app_service.get_prompt_detail(-1)
    return message, detail["comments"], ""


def load_prompt_data() -> List[PromptRecord]:
    """
    Tries Excel first, then falls back to sample seed data.
    """
    try:
        excel_records = PromptExcelLoader.load_from_excel(EXCEL_FILE_PATH)
        if excel_records:
            return excel_records
    except Exception as exc:
        print(f"[WARN] Failed to load Excel data: {exc}")

    print("[INFO] Falling back to bundled sample prompt data.")
    return get_seed_prompts()


# ============================================================
# App Initialization
# ============================================================

repository = PromptRepository(seed_data=load_prompt_data())
app_service = PromptService(repository)

custom_css = """
#app-shell {
    max-width: 1080px;
    margin: 0 auto;
}
.section-note {
    color: #666;
    font-size: 0.92rem;
}
textarea, input {
    font-size: 14px !important;
}
"""

minimal_theme = gr.themes.Base(
    primary_hue=colors.gray,
    secondary_hue=colors.slate,
    neutral_hue=colors.gray,
    spacing_size=sizes.spacing_md,
    radius_size=sizes.radius_sm,
    text_size=sizes.text_md,
)


# ============================================================
# UI
# ============================================================

with gr.Blocks(title="Banking Analytics Prompt Library", theme=minimal_theme, css=custom_css) as demo:
    with gr.Column(elem_id="app-shell"):
        gr.Markdown("""
        # Banking Analytics Prompt Library
        <div class="section-note">A clean, searchable library of reusable prompts for banking analytics.</div>
        """)

        with gr.Row():
            search_bar = gr.Textbox(
                label="Search",
                placeholder="Search by prompt name, tag, category, or content",
                scale=3,
            )
            category_filter = gr.Dropdown(
                label="Category",
                choices=repository.categories(),
                value="All",
                scale=1,
            )

        prompt_selector = gr.Dropdown(
            label="Prompt",
            choices=build_prompt_dropdown_choices(app_service, "", "All"),
            value=repository.list_all()[0].id if repository.list_all() else None,
            interactive=True,
        )

        with gr.Row():
            with gr.Column(scale=1):
                prompt_name = gr.Textbox(label="Name", interactive=False)
                prompt_category = gr.Textbox(label="Category", interactive=False)
                prompt_tags = gr.Textbox(label="Tags", interactive=False)
                prompt_objective = gr.Textbox(label="Prompt Objective", lines=4, interactive=False)
                required_inputs = gr.Textbox(label="Required Inputs", lines=5, interactive=False)
                optional_inputs = gr.Textbox(label="Optional Inputs", lines=4, interactive=False)
                expected_result = gr.Textbox(label="Sample Output", lines=6, interactive=False)

            with gr.Column(scale=1):
                prompt_visualizer = gr.Textbox(
                    label="Prompt View",
                    lines=16,
                    interactive=False,
                )
                copy_visualizer_btn = gr.Button("Copy View", size="sm")
                copy_payload = gr.Textbox(
                    label="Prompt Template",
                    lines=12,
                    interactive=False,
                )
                copy_payload_btn = gr.Button("Copy Prompt", size="sm")

        with gr.Row():
            with gr.Column(scale=1):
                rating_summary = gr.Textbox(label="Ratings", interactive=False)
                rating_input = gr.Slider(
                    label="Rate Prompt",
                    minimum=1,
                    maximum=5,
                    step=1,
                    value=5,
                )
                rate_button = gr.Button("Submit Rating", variant="secondary")
                rating_message = gr.Textbox(label="Rating Status", interactive=False)

            with gr.Column(scale=1):
                comments_box = gr.Textbox(label="Comments", lines=8, interactive=False)
                comment_input = gr.Textbox(
                    label="Add Comment",
                    placeholder="Share feedback or usage notes",
                    lines=3,
                )
                comment_button = gr.Button("Add Comment", variant="secondary")
                comment_message = gr.Textbox(label="Comment Status", interactive=False)

        copy_feedback = gr.Textbox(label="Copy Status", interactive=False)

    initial_id = repository.list_all()[0].id if repository.list_all() else None

    demo.load(
        fn=lambda: render_prompt_details(initial_id),
        outputs=[
            prompt_name,
            prompt_category,
            prompt_tags,
            prompt_objective,
            required_inputs,
            optional_inputs,
            expected_result,
            rating_summary,
            comments_box,
            prompt_visualizer,
            copy_payload,
        ],
    )

    search_bar.change(
        fn=refresh_prompt_list,
        inputs=[search_bar, category_filter],
        outputs=[
            prompt_selector,
            prompt_name,
            prompt_category,
            prompt_tags,
            prompt_objective,
            required_inputs,
            optional_inputs,
            expected_result,
            rating_summary,
            comments_box,
            prompt_visualizer,
            copy_payload,
        ],
    )

    category_filter.change(
        fn=refresh_prompt_list,
        inputs=[search_bar, category_filter],
        outputs=[
            prompt_selector,
            prompt_name,
            prompt_category,
            prompt_tags,
            prompt_objective,
            required_inputs,
            optional_inputs,
            expected_result,
            rating_summary,
            comments_box,
            prompt_visualizer,
            copy_payload,
        ],
    )

    prompt_selector.change(
        fn=render_prompt_details,
        inputs=[prompt_selector],
        outputs=[
            prompt_name,
            prompt_category,
            prompt_tags,
            prompt_objective,
            required_inputs,
            optional_inputs,
            expected_result,
            rating_summary,
            comments_box,
            prompt_visualizer,
            copy_payload,
        ],
    )

    rate_button.click(
        fn=handle_rating,
        inputs=[prompt_selector, rating_input],
        outputs=[rating_message, rating_summary],
    )

    comment_button.click(
        fn=handle_comment,
        inputs=[prompt_selector, comment_input],
        outputs=[comment_message, comments_box, comment_input],
    )

    copy_visualizer_btn.click(
        None,
        inputs=[prompt_visualizer],
        outputs=[],
        js="""
        (value) => {
            navigator.clipboard.writeText(value || '');
            return [];
        }
        """,
    ).then(
        fn=lambda: "Prompt view copied.",
        outputs=[copy_feedback],
    )

    copy_payload_btn.click(
        None,
        inputs=[copy_payload],
        outputs=[],
        js="""
        (value) => {
            navigator.clipboard.writeText(value || '');
            return [];
        }
        """,
    ).then(
        fn=lambda: "Prompt template copied.",
        outputs=[copy_feedback],
    )


if __name__ == "__main__":
    demo.launch()

C:\Users\MohitVerma\AppData\Local\Temp\ipykernel_74392\2901914611.py:528: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Banking Analytics Prompt Library", theme=minimal_theme, css=custom_css) as demo:


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
